In [ ]:
print("RAG IMPLEMENTATION")

In [ ]:
%pip install python-dotenv pydantic chromadb tqdm numpy scikit-learn plotly sentence-transformers

In [ ]:
# %pip install chromadb

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
import importlib.util

# if importlib.util.find_spec("sentence_transformers") is None:
#     %pip install sentence-transformers

from sentence_transformers import SentenceTransformer


In [7]:
load_dotenv(override=True)
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "gpt-oss:120b-cloud" 
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
KNOWLEDGE_BASE_PATH = Path("knowledge_base")
AVERAGE_CHUNK_SIZE = 500
openai = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8525.68it/s]


In [8]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [10]:
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")
    
    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)
    
class Chunks(BaseModel):
    chunks: list[Chunk]


In [20]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [21]:
documents = fetch_documents()

Loaded 2 documents


In [22]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    # print(how_many)
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [23]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: data
The document has been retrieved from: knowledge_base/data/lost_library.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 4 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# The Lost Library of Arkania

## Chapter 1: The Di

In [24]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [25]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: data\nThe document has been retrieved from: knowledge_base/data/lost_library.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don\'t leave anything out.\nThis document should probably be split into 4 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n#